# PayPal Credit Agent — Telegram Bot
Run this cell to start the bot. Open `t.me/PayPalCreditAgentBot` on Telegram to test.

**Stop the cell to shut down the bot.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PayPal Credit Agent — Single-cell Telegram Bot for Google Colab
# ═══════════════════════════════════════════════════════════════

!pip install -q python-telegram-bot

import asyncio, re, logging
from enum import Enum
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import (
    ApplicationBuilder, CommandHandler, CallbackQueryHandler,
    MessageHandler, ContextTypes, filters,
)
from telegram.constants import ChatAction

logging.basicConfig(level=logging.INFO)

# ── CONFIG ─────────────────────────────────────
BOT_TOKEN = "8703693982:AAF4baQTOuXbYM2F42ne-9HPIQ5vqjHJrFM"

# ── STATE ──────────────────────────────────────
class State(str, Enum):
    IDLE = "idle"
    GREETED = "greeted"
    OFFERS_SHOWN = "offers_shown"
    SELECTED = "selected"
    APPROVED = "approved"

sessions = {}
def get(uid): 
    if uid not in sessions:
        sessions[uid] = {"state": State.IDLE, "offer": None}
    return sessions[uid]
def reset(uid): sessions[uid] = {"state": State.IDLE, "offer": None}

# ── MOCK DATA ─────────────────────────────────
USER = {"name": "Arun Sharma", "email": "arun.sharma@email.com",
        "tenure": 36, "band": "prime", "spend": 4200}

OFFERS = [
    {"tag": "Best Match", "name": "PayPal Pay Later",
     "amt": "$2,500", "score": 96, "detail": "0% APR · 6 months · No annual fee"},
    {"tag": "Premium", "name": "PayPal Cashback Mastercard",
     "amt": "$5,000", "score": 84, "detail": "3% cashback on all PayPal purchases"},
    {"tag": "Starter", "name": "PayPal Credit Line",
     "amt": "$1,200", "score": 71, "detail": "Build credit history · 19.99% APR"},
]

TXNS = [
    ("👟", "Nike.com", "Sports", "-$129.00", "Apr 1"),
    ("🍔", "Uber Eats", "Food", "-$24.50", "Mar 30"),
    ("📦", "Amazon", "Shopping", "-$67.99", "Mar 28"),
    ("☕", "Starbucks", "Coffee", "-$8.75", "Mar 27"),
    ("🎵", "Spotify", "Subscription", "-$9.99", "Mar 25"),
    ("🛒", "Target", "Shopping", "-$45.20", "Mar 23"),
    ("🎬", "Netflix", "Subscription", "-$15.99", "Mar 20"),
    ("💳", "PayPal Credit", "Payment", "+$200.00", "Mar 15"),
]

# ── KEYBOARDS ──────────────────────────────────
def menu_kb():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton("💳 Apply for Credit", callback_data="topic:credit"),
         InlineKeyboardButton("💰 Check Balance", callback_data="topic:balance")],
        [InlineKeyboardButton("🎁 View Rewards", callback_data="topic:rewards"),
         InlineKeyboardButton("🙋 Support", callback_data="topic:support")],
    ])

def offers_kb():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton(f"{'⭐ ' if o['score']>90 else ''}{o['name']} — {o['amt']}",
         callback_data=f"offer:{i}")] for i, o in enumerate(OFFERS)
    ])

def confirm_kb():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton("✅ Submit Application", callback_data="action:submit")],
        [InlineKeyboardButton("← Back to Offers", callback_data="action:back")],
    ])

def post_kb():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton("📋 Statement", callback_data="action:statement"),
         InlineKeyboardButton("🃏 Manage Card", callback_data="action:card")],
        [InlineKeyboardButton("💰 Balance", callback_data="topic:balance"),
         InlineKeyboardButton("🎁 Rewards", callback_data="topic:rewards")],
    ])

def support_kb():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton("📞 Call PayPal", callback_data="sup:call")],
        [InlineKeyboardButton("💬 Live Chat", callback_data="sup:chat")],
        [InlineKeyboardButton("⚠️ Dispute a Charge", callback_data="sup:dispute")],
        [InlineKeyboardButton("🔒 Lost Card", callback_data="sup:lost")],
    ])

# ── MESSAGES ───────────────────────────────────
def welcome_msg():
    return ("👋 *Welcome to PayPal Credit Agent*\n\n"
            "I can help you with credit products, check your balance, "
            "view rewards, and more.\n\n"
            "Choose an option below or just type your question:")

def scoring_msg():
    u = USER
    return (f"🧠 *Analyzing your profile...*\n\n"
            f"👤 {u['name']}\n📧 {u['email']}\n"
            f"📅 PayPal member: {u['tenure']} months\n"
            f"💳 Credit band: _{u['band']}_\n"
            f"💰 Avg monthly spend: ${u['spend']:,}\n\n"
            f"_Running NBA model..._")

def offers_msg():
    lines = ["🎯 *Credit Offers for You*\n"]
    for o in OFFERS:
        star = "⭐ " if o['score'] > 90 else ""
        lines.append(f"{star}*{o['name']}* — {o['amt']}")
        lines.append(f"   {o['detail']}")
        lines.append(f"   Match: {o['score']}% | _{o['tag']}_\n")
    lines.append("Select an offer below:")
    return "\n".join(lines)

def confirm_msg(i):
    o = OFFERS[i]
    return (f"✅ *Application Ready*\n━━━━━━━━━━━━━━━━━\n"
            f"Product: *{o['name']}*\nCredit Limit: *{o['amt']}*\n"
            f"Applicant: {USER['name']}\nChannel: Telegram\n"
            f"Decision: _Instant · ~3s_\n\nTap Submit to apply:")

def approval_msg(i):
    o = OFFERS[i]
    return (f"🎉 *Congratulations!*\n\n"
            f"Your *{o['name']}* has been approved!\n\n"
            f"💳 Credit Limit: *{o['amt']}*\n"
            f"⏱ Decision Time: *3.1 seconds*\n"
            f"📋 Status: _Active_\n\nWhat would you like to do next?")

def balance_msg():
    return ("💰 *Account Balance*\n━━━━━━━━━━━━━━━━━\n"
            "Current Balance: *$847.23*\nAvailable Credit: *$1,652.77*\n"
            "Credit Limit: $2,500\nDue Date: Apr 15\n"
            "Min Payment: $25.00\nUtilization: 33.9%")

def statement_msg():
    lines = ["📋 *Recent Transactions*\n━━━━━━━━━━━━━━━━━\n"]
    for icon, name, cat, amt, dt in TXNS:
        cr = " 💚" if amt.startswith("+") else ""
        lines.append(f"{icon} *{name}*  {amt}{cr}")
        lines.append(f"   _{cat}_ · {dt}\n")
    return "\n".join(lines)

def rewards_msg():
    return ("🎁 *Your Rewards*\n━━━━━━━━━━━━━━━━━\n"
            "Cash Back YTD: *$114.20*\nPoints Balance: *42,180*\n"
            "Tier: Silver\nNext Milestone: _7,820 pts to Gold_")

def card_msg():
    return ("🃏 *Card Management*\n━━━━━━━━━━━━━━━━━\n"
            "Card: •••• •••• •••• 4821\nHolder: ARUN SHARMA\n"
            "Expiry: 09/28\nType: PayPal Pay Later\n\n"
            "*Controls:*\n🌐 Online Purchases: ✅ On\n"
            "✈️ International: ❌ Off\n📱 Contactless: ✅ On\n"
            "🔔 Spend Alerts: ✅ On\n\n"
            "*Quick Actions:*\n🧊 Freeze | 🔄 Replace | ⚠️ Report | 🔑 PIN")

# ── INTENT DETECTION ──────────────────────────
INTENTS = [
    ("card_manage", r"manage|card settings|card details|controls|settings|freeze|block|lock|suspend|replace|lost card"),
    ("score", r"credit score|fico|creditworthiness|credit rating"),
    ("credit", r"apply|credit card|loan|borrow|offer|limit increase|get credit|pay later|cashback master|credit line"),
    ("balance", r"balance|owe|due date|minimum|available credit|statement summary"),
    ("statement", r"statement|transactions|spending|history|recent|charges|purchases"),
    ("rewards", r"reward|cashback|points|miles|earn|redeem|upgrade"),
    ("limit", r"limit|how much left|remaining|budget|utilis|utiliz"),
    ("pay", r"pay|make payment"),
    ("support", r"support|contact|help|issue|problem|talk human|agent|representative|dispute|call paypal"),
    ("menu", r"^hi$|^hello$|what can|commands|options|menu|start|home"),
    ("thanks", r"thanks|thank you|thx|great|awesome|perfect|nice|got it|^ok$|^okay$"),
]

def detect(text):
    t = text.lower().strip()
    for intent, pat in INTENTS:
        if re.search(pat, t): return intent
    return None

# ── HANDLERS ───────────────────────────────────
async def cmd_start(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    reset(update.effective_user.id)
    get(update.effective_user.id)["state"] = State.GREETED
    await update.message.reply_text(welcome_msg(), parse_mode="Markdown", reply_markup=menu_kb())

async def cmd_menu(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("Choose an option:", reply_markup=menu_kb())

async def cmd_help(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "*PayPal Credit Agent — Help*\n\n"
        "💳 /start — Start fresh\n📋 /menu — Show menu\n"
        "↺ /reset — Reset session\n❓ /help — This message\n\n"
        "Or type naturally:\n"
        '• _\"I want a credit card\"_\n'
        '• _\"Show my balance\"_\n'
        '• _\"View transactions\"_',
        parse_mode="Markdown")

async def cmd_reset(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    reset(update.effective_user.id)
    await update.message.reply_text("↺ Session reset. Type /start to begin again.")

async def on_callback(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    q = update.callback_query
    await q.answer()
    uid = q.from_user.id
    s = get(uid)
    data = q.data

    if data == "topic:credit":
        await q.message.chat.send_action(ChatAction.TYPING)
        await q.message.reply_text(scoring_msg(), parse_mode="Markdown")
        await asyncio.sleep(2)
        s["state"] = State.OFFERS_SHOWN
        await q.message.reply_text(offers_msg(), parse_mode="Markdown", reply_markup=offers_kb())

    elif data == "topic:balance":
        await q.message.reply_text(balance_msg(), parse_mode="Markdown")

    elif data == "topic:rewards":
        await q.message.reply_text(rewards_msg(), parse_mode="Markdown")

    elif data == "topic:support":
        await q.message.reply_text("🙋 *How can we help?*", parse_mode="Markdown", reply_markup=support_kb())

    elif data.startswith("offer:"):
        i = int(data.split(":")[1])
        s["offer"] = i
        s["state"] = State.SELECTED
        await q.message.reply_text(confirm_msg(i), parse_mode="Markdown", reply_markup=confirm_kb())

    elif data == "action:submit":
        i = s.get("offer", 0)
        await q.message.chat.send_action(ChatAction.TYPING)
        await q.message.reply_text("⏳ _Processing your application..._", parse_mode="Markdown")
        await asyncio.sleep(2)
        s["state"] = State.APPROVED
        await q.message.reply_text(approval_msg(i), parse_mode="Markdown", reply_markup=post_kb())

    elif data == "action:back":
        s["state"] = State.OFFERS_SHOWN
        await q.message.reply_text(offers_msg(), parse_mode="Markdown", reply_markup=offers_kb())

    elif data == "action:statement":
        await q.message.reply_text(statement_msg(), parse_mode="Markdown")

    elif data == "action:card":
        await q.message.reply_text(card_msg(), parse_mode="Markdown")

    elif data.startswith("sup:"):
        msgs = {
            "call": "📞 *Call PayPal*\n\n1-888-221-1161\nMon–Fri 6am–6pm PT",
            "chat": "💬 *Live Chat*\n\nVisit: paypal.com/us/smarthelp/home",
            "dispute": "⚠️ *Dispute a Charge*\n\n1. Go to Activity\n2. Find the transaction\n3. Tap 'Report a Problem'",
            "lost": "🔒 *Lost Card*\n\nYour card has been frozen for safety.\nReplacement in 3-5 business days.",
        }
        await q.message.reply_text(msgs.get(data.split(":")[1], "Contact 1-888-221-1161"), parse_mode="Markdown")

async def on_message(update: Update, ctx: ContextTypes.DEFAULT_TYPE):
    text = update.message.text
    if not text: return
    uid = update.effective_user.id
    s = get(uid)
    intent = detect(text)
    await update.message.chat.send_action(ChatAction.TYPING)

    if intent == "credit":
        s["state"] = State.OFFERS_SHOWN
        await update.message.reply_text(scoring_msg(), parse_mode="Markdown")
        await asyncio.sleep(2)
        await update.message.reply_text(offers_msg(), parse_mode="Markdown", reply_markup=offers_kb())
    elif intent in ("balance", "limit"):
        await update.message.reply_text(balance_msg(), parse_mode="Markdown")
    elif intent == "statement":
        await update.message.reply_text(statement_msg(), parse_mode="Markdown")
    elif intent == "rewards":
        await update.message.reply_text(rewards_msg(), parse_mode="Markdown")
    elif intent == "card_manage":
        await update.message.reply_text(card_msg(), parse_mode="Markdown")
    elif intent == "score":
        await update.message.reply_text(
            "📊 *Credit Score*\n━━━━━━━━━━━━━━━━━\n"
            "FICO Score: *742*\nRating: _Excellent_\nLast updated: Apr 1, 2026",
            parse_mode="Markdown")
    elif intent == "pay":
        await update.message.reply_text(
            "💳 *Make a Payment*\n\nCurrent balance: *$847.23*\n"
            "Min payment: $25.00\nDue: Apr 15\n\n_Payment feature coming soon._",
            parse_mode="Markdown")
    elif intent == "support":
        await update.message.reply_text("🙋 *How can we help?*", parse_mode="Markdown", reply_markup=support_kb())
    elif intent == "menu":
        await update.message.reply_text(welcome_msg(), parse_mode="Markdown", reply_markup=menu_kb())
    elif intent == "thanks":
        await update.message.reply_text("You're welcome! 😊 Type /menu if you need anything else.")
    else:
        await update.message.reply_text("I'm not sure I understand. Here's what I can help with:", reply_markup=menu_kb())

# ── RUN BOT ────────────────────────────────────
print("🚀 Starting PayPal Credit Agent bot...")
print("📱 Open t.me/PayPalCreditAgentBot on Telegram and tap Start")
print("⏹ Press the STOP button in Colab to shut down\n")

app = ApplicationBuilder().token(BOT_TOKEN).build()
app.add_handler(CommandHandler("start", cmd_start))
app.add_handler(CommandHandler("menu", cmd_menu))
app.add_handler(CommandHandler("help", cmd_help))
app.add_handler(CommandHandler("reset", cmd_reset))
app.add_handler(CallbackQueryHandler(on_callback))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, on_message))

# Colab-compatible run (nest_asyncio not needed with run_polling)
app.run_polling(drop_pending_updates=True)
